# Lab — Modern CNN architectures and deployment-aware selection

You are selecting a visual backbone for the same inspection task under three deployment contracts: a cloud GPU service, a factory CPU workstation, and an edge camera. You will not crown one universal winner. You will build evidence that makes a conditional decision defensible.

This notebook is self-contained: every model, data generator, profiler, test, plot, and export is defined here. It uses five real pretrained encoders from the official `torchvision` SDK:

- ResNet-18 and ResNet-50;
- MobileNetV3-Large;
- EfficientNet-B0; and
- ConvNeXt-Tiny.

The default path is intentionally bounded and CPU-safe. Set `CV_FULL_RUN=1` **before** starting the kernel for more samples, the 128/224/320 resolution sweep, longer timing loops, and more fine-tuning epochs.

## 0. Experiment contract

We will hold the task, source-aware split, classifier family, preprocessing policy, and random seed constant while changing the encoder. The frozen-probe phase is the controlled architecture comparison. Fine-tuning is a separate experiment because it changes the representation and optimization budget.

Success means:

1. the test set comes only from an unseen factory;
2. every frozen encoder uses the same logistic-regression probe;
3. latency uses warm-up, repeats, synchronization, and tail percentiles;
4. quality includes defect recall and shifted performance, not only accuracy;
5. a Pareto and contract analysis replaces a universal rank; and
6. the result records the hardware and software on which it was measured.

In [ ]:
from __future__ import annotations

import copy
import io
import json
import math
import os
import platform
import random
import statistics
import time
import zlib
from collections import OrderedDict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    pairwise_distances,
    recall_score,
    silhouette_score,
)
from sklearn.preprocessing import normalize
from torch.profiler import ProfilerActivity, profile
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import (
    ConvNeXt_Tiny_Weights,
    EfficientNet_B0_Weights,
    MobileNet_V3_Large_Weights,
    ResNet18_Weights,
    ResNet50_Weights,
    convnext_tiny,
    efficientnet_b0,
    mobilenet_v3_large,
    resnet18,
    resnet50,
)

SEED = 23
FULL_RUN = os.getenv("CV_FULL_RUN", "0") == "1"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

torch.set_grad_enabled(True)
CLASS_NAMES = ["Normal", "Micro Scratch", "Structural Crack", "Contamination"]
DEFECT_IDS = [1, 2, 3]
COURSE_DIR = Path.cwd()
if COURSE_DIR.name != "02-modern-cnn-architectures-efficient-vision":
    candidate = Path.cwd() / "curriculum/beginner/02-modern-cnn-architectures-efficient-vision"
    if candidate.exists():
        COURSE_DIR = candidate
ARTIFACT_DIR = COURSE_DIR / ".artifacts/architecture_benchmark"
DATA_DIR = ARTIFACT_DIR / "dataset"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

RUN = {
    "full_run": FULL_RUN,
    "device": str(DEVICE),
    "train_per_class_per_source": 16 if FULL_RUN else 6,
    "val_per_class_per_source": 6 if FULL_RUN else 3,
    "test_per_class": 10 if FULL_RUN else 4,
    "benchmark_resolution": 224 if FULL_RUN else 128,
    "resolution_sweep": [128, 224, 320] if FULL_RUN else [96, 128, 160],
    "timing_repeats": 30 if FULL_RUN else 8,
    "fine_tune_epochs": 3 if FULL_RUN else 1,
}

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "processor": platform.processor() or "not reported",
    "torch": torch.__version__,
    "torchvision": __import__("torchvision").__version__,
    "scikit_learn": sklearn.__version__,
    "device": str(DEVICE),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps({"run": RUN, "environment": environment}, indent=2))

## 1. Write the deployment contracts first

Metrics become easy to manipulate when constraints are invented after seeing results. These teaching contracts are declared before benchmarking. Their numbers are intentionally editable; they are not claims about a particular production device.

In [ ]:
CONTRACTS = OrderedDict({
    "Cloud GPU service": {
        "hard": {"robust_f1_min": 0.55, "model_mb_max": 140.0},
        "optimize": ["robust_f1:max", "throughput_b8:max"],
        "note": "Prefer robust quality, then batched throughput within the memory envelope.",
    },
    "Factory CPU workstation": {
        "hard": {"defect_recall_min": 0.60, "p90_ms_b1_max": 1500.0},
        "optimize": ["defect_recall:max", "median_ms_b1:min"],
        "note": "Protect defect recall while keeping batch-1 latency bounded.",
    },
    "Edge camera": {
        "hard": {"model_mb_max": 30.0, "p90_ms_b1_max": 800.0, "defect_recall_min": 0.45},
        "optimize": ["median_ms_b1:min", "robust_f1:max"],
        "note": "Treat package size and predictable batch-1 execution as hard limits.",
    },
})

pd.DataFrame([
    {"deployment": name, "hard constraints": json.dumps(spec["hard"]),
     "optimization order": " → ".join(spec["optimize"]), "intent": spec["note"]}
    for name, spec in CONTRACTS.items()
]).style.hide(axis="index")

## 2. Generate a source-aware inspection dataset

The dataset is synthetic so the entire lab is reproducible, but its split models a real deployment boundary. Factories A and B contribute train/validation examples. Factory C is test-only. Source affects lighting, tint, placement, and sensor noise. Defect scale is recorded so the resolution study can isolate small-defect recall.

Synthetic results teach experimental mechanics; they do not estimate production accuracy.

In [ ]:
def make_component(label_id: int, source: str, sample_seed: int, size: int = 192) -> tuple[Image.Image, bool]:
    rng = np.random.default_rng(sample_seed)
    source_cfg = {
        "Factory_A": {"bg": (39, 49, 58), "metal": (178, 184, 188), "tint": (4, 0, 0), "noise": 2.0},
        "Factory_B": {"bg": (47, 43, 39), "metal": (184, 177, 167), "tint": (0, 3, 5), "noise": 3.0},
        "Factory_C": {"bg": (31, 48, 46), "metal": (163, 181, 176), "tint": (0, 6, 2), "noise": 4.5},
    }[source]
    yy, xx = np.mgrid[:size, :size]
    base = np.zeros((size, size, 3), dtype=np.float32)
    base[:] = source_cfg["bg"]
    base += ((xx / size) * 8 - (yy / size) * 5)[..., None]
    base += np.asarray(source_cfg["tint"])[None, None, :]
    base += rng.normal(0, source_cfg["noise"], base.shape)
    image = Image.fromarray(np.uint8(np.clip(base, 0, 255)))
    draw = ImageDraw.Draw(image)

    jitter = rng.integers(-8, 9, size=2)
    cx, cy = size // 2 + int(jitter[0]), size // 2 + int(jitter[1])
    radius = int(size * 0.32)
    metal = source_cfg["metal"]
    draw.rounded_rectangle((cx-radius, cy-radius, cx+radius, cy+radius), radius=18,
                           fill=metal, outline=(220, 224, 225), width=3)
    draw.ellipse((cx-17, cy-17, cx+17, cy+17), fill=(72, 78, 81), outline=(230, 232, 232), width=3)
    for angle in np.linspace(0, 2*np.pi, 6, endpoint=False):
        bx, by = cx + int(radius*0.72*np.cos(angle)), cy + int(radius*0.72*np.sin(angle))
        draw.ellipse((bx-5, by-5, bx+5, by+5), fill=(75, 80, 82), outline=(220, 222, 222))

    small_defect = bool(sample_seed % 2)
    if label_id == 1:  # micro scratch
        length = 18 if small_defect else 42
        x0, y0 = cx - length//2, cy + int(rng.integers(-28, 29))
        draw.line((x0, y0, x0+length, y0+int(rng.integers(-4, 5))), fill=(66, 53, 48), width=1 if small_defect else 2)
    elif label_id == 2:  # structural crack
        length = 24 if small_defect else 50
        pts = [(cx-length//2, cy-12)]
        for step in range(1, 6):
            pts.append((cx-length//2 + step*length//5, cy-12 + int(rng.integers(-8, 9))))
        draw.line(pts, fill=(35, 35, 38), width=2 if small_defect else 4)
        if not small_defect:
            draw.line((pts[3][0], pts[3][1], pts[3][0]+12, pts[3][1]+14), fill=(38, 37, 39), width=2)
    elif label_id == 3:  # contamination
        count = 2 if small_defect else 5
        for _ in range(count):
            x, y = cx + int(rng.integers(-45, 46)), cy + int(rng.integers(-45, 46))
            r = int(rng.integers(3, 7) if small_defect else rng.integers(7, 13))
            draw.ellipse((x-r, y-r, x+r, y+r), fill=(108, 86, 40), outline=(80, 63, 31))
    else:
        small_defect = False

    image = image.filter(ImageFilter.GaussianBlur(radius=0.25 if source != "Factory_C" else 0.55))
    return image, small_defect


def generate_dataset() -> pd.DataFrame:
    records = []
    split_plan = [
        ("train", "Factory_A", RUN["train_per_class_per_source"]),
        ("train", "Factory_B", RUN["train_per_class_per_source"]),
        ("val", "Factory_A", RUN["val_per_class_per_source"]),
        ("val", "Factory_B", RUN["val_per_class_per_source"]),
        ("test", "Factory_C", RUN["test_per_class"]),
    ]
    for split_idx, (split, source, count) in enumerate(split_plan):
        for label_id, label in enumerate(CLASS_NAMES):
            for sample_idx in range(count):
                sample_seed = SEED * 10_000 + split_idx * 1_000 + label_id * 100 + sample_idx
                image, small = make_component(label_id, source, sample_seed)
                filename = f"{split}_{source}_{label_id}_{sample_idx:03d}.png"
                path = DATA_DIR / filename
                image.save(path)
                records.append({"path": str(path), "split": split, "source": source,
                                "label": label, "label_id": label_id,
                                "small_defect": bool(small and label_id in DEFECT_IDS),
                                "sample_seed": sample_seed})
    return pd.DataFrame(records)


data = generate_dataset()
assert set(data.query("split == 'test'").source) == {"Factory_C"}
assert not set(data.query("split == 'train'").path) & set(data.query("split == 'test'").path)
display(pd.crosstab([data.split, data.source], data.label))
print(f"Generated {len(data)} images at {DATA_DIR}")

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(10, 10))
for row, label in enumerate(CLASS_NAMES):
    examples = data.query("label == @label").groupby("source", group_keys=False).head(2).head(4)
    for col, (_, record) in enumerate(examples.iterrows()):
        axes[row, col].imshow(Image.open(record.path))
        axes[row, col].set_title(f"{record.source}\nsmall={record.small_defect}", fontsize=8)
        axes[row, col].axis("off")
plt.suptitle("Same label family across sources and defect scales")
plt.tight_layout()
plt.show()

## 3. Residual learning: an optimization experiment

The next experiment fits a fixed synthetic image-to-image mapping with comparable-width networks. It is an illustration, not proof that every deep plain network fails. Watch training loss and the first layer’s gradient norm. A deeper plain stack has the capacity to reproduce a shallow solution, but the parameterization can make that solution harder to find.

In [ ]:
class PlainMapper(nn.Module):
    def __init__(self, depth: int, width: int = 16):
        super().__init__()
        layers = [nn.Conv2d(3, width, 3, padding=1), nn.ReLU()]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(width, width, 3, padding=1), nn.ReLU()]
        layers += [nn.Conv2d(width, 3, 3, padding=1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class ResidualUnit(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        self.branch = nn.Sequential(
            nn.Conv2d(width, width, 3, padding=1), nn.ReLU(),
            nn.Conv2d(width, width, 3, padding=1),
        )

    def forward(self, x):
        return torch.relu(x + self.branch(x))


class ResidualMapper(nn.Module):
    def __init__(self, blocks: int = 4, width: int = 16):
        super().__init__()
        self.stem = nn.Conv2d(3, width, 3, padding=1)
        self.blocks = nn.Sequential(*[ResidualUnit(width) for _ in range(blocks)])
        self.head = nn.Conv2d(width, 3, 3, padding=1)

    def forward(self, x):
        return self.head(self.blocks(torch.relu(self.stem(x))))


demo_x = torch.rand(48, 3, 16, 16, device=DEVICE)
demo_target = 0.7 * demo_x + 0.3 * torch.nn.functional.avg_pool2d(demo_x, 3, stride=1, padding=1)
models_demo = OrderedDict({
    "plain-shallow": PlainMapper(depth=3).to(DEVICE),
    "plain-deep": PlainMapper(depth=10).to(DEVICE),
    "residual-deep": ResidualMapper(blocks=4).to(DEVICE),
})

history = {}
steps = 80 if FULL_RUN else 35
for name, model in models_demo.items():
    torch.manual_seed(SEED)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    losses, gradients = [], []
    for _ in range(steps):
        optimizer.zero_grad(set_to_none=True)
        loss = torch.nn.functional.mse_loss(model(demo_x), demo_target)
        loss.backward()
        first_weight = next(p for p in model.parameters() if p.ndim == 4)
        gradients.append(float(first_weight.grad.norm().detach().cpu()))
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    history[name] = {"loss": losses, "gradient": gradients,
                     "parameters": sum(p.numel() for p in model.parameters())}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, values in history.items():
    axes[0].plot(values["loss"], label=f"{name} ({values['parameters']:,} params)")
    axes[1].plot(values["gradient"], label=name)
axes[0].set(title="Optimization on the same fixed mapping", xlabel="step", ylabel="MSE", yscale="log")
axes[1].set(title="First-convolution gradient norm", xlabel="step", ylabel="L2 norm", yscale="log")
for ax in axes: ax.legend(fontsize=8); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

pd.DataFrame([{"model": n, "parameters": v["parameters"], "final_loss": v["loss"][-1],
               "first_gradient": v["gradient"][0]} for n, v in history.items()]).style.hide(axis="index")

## 4. Inspect the building blocks

A basic residual block uses two spatial convolutions. A bottleneck moves the expensive spatial operation into a lower-dimensional channel space. A projection shortcut is required when shape or stride changes.

In [ ]:
class BasicResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.branch = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = (nn.Identity() if in_channels == out_channels and stride == 1 else
                         nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False))

    def forward(self, x):
        return torch.relu(self.branch(x) + self.shortcut(x))


class BottleneckBlock(nn.Module):
    def __init__(self, in_channels: int, bottleneck: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.branch = nn.Sequential(
            nn.Conv2d(in_channels, bottleneck, 1, bias=False), nn.ReLU(),
            nn.Conv2d(bottleneck, bottleneck, 3, stride=stride, padding=1, bias=False), nn.ReLU(),
            nn.Conv2d(bottleneck, out_channels, 1, bias=False),
        )
        self.shortcut = (nn.Identity() if in_channels == out_channels and stride == 1 else
                         nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False))

    def forward(self, x):
        return torch.relu(self.branch(x) + self.shortcut(x))


x = torch.randn(2, 64, 56, 56)
basic = BasicResidualBlock(64, 128, stride=2)
bottle = BottleneckBlock(64, 32, 128, stride=2)
manual_bottleneck_branch = 64*32 + 3*3*32*32 + 32*128
measured_bottleneck_branch = sum(p.numel() for p in bottle.branch.parameters())
assert measured_bottleneck_branch == manual_bottleneck_branch

pd.DataFrame([
    {"block": "basic + projection", "output": tuple(basic(x).shape),
     "parameters": sum(p.numel() for p in basic.parameters())},
    {"block": "bottleneck + projection", "output": tuple(bottle(x).shape),
     "parameters": sum(p.numel() for p in bottle.parameters())},
]).style.hide(axis="index")

### Dense, grouped, and depthwise-separable convolution

The table verifies parameter and approximate MAC formulas, then times the actual operators on this runtime. Fewer operations can still lose if the backend executes a shape or grouped kernel inefficiently.

In [ ]:
def sync_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        torch.mps.synchronize()


def median_module_ms(module: nn.Module, sample: torch.Tensor, warmup: int = 3, repeats: int = 15) -> float:
    module = module.to(DEVICE).eval()
    sample = sample.to(DEVICE)
    with torch.inference_mode():
        for _ in range(warmup): module(sample)
        sync_device()
        values = []
        for _ in range(repeats):
            sync_device(); start = time.perf_counter()
            module(sample)
            sync_device(); values.append((time.perf_counter() - start) * 1_000)
    return statistics.median(values)


cin, cout, side, kernel = 32, 64, 56, 3
operator_cases = OrderedDict({
    "dense": nn.Conv2d(cin, cout, kernel, padding=1, bias=False),
    "grouped-4": nn.Conv2d(cin, cout, kernel, padding=1, groups=4, bias=False),
    "depthwise + pointwise": nn.Sequential(
        nn.Conv2d(cin, cin, kernel, padding=1, groups=cin, bias=False),
        nn.Conv2d(cin, cout, 1, bias=False),
    ),
})
manual_params = {
    "dense": kernel*kernel*cin*cout,
    "grouped-4": kernel*kernel*cin*cout//4,
    "depthwise + pointwise": kernel*kernel*cin + cin*cout,
}
sample = torch.randn(1, cin, side, side)
operator_rows = []
for name, module in operator_cases.items():
    actual = sum(p.numel() for p in module.parameters())
    assert actual == manual_params[name]
    operator_rows.append({"operator": name, "parameters": actual,
                          "approx_macs_m": side*side*actual/1e6,
                          f"median_ms_on_{DEVICE.type}": median_module_ms(module, sample)})
operator_df = pd.DataFrame(operator_rows)
operator_df.style.hide(axis="index").format(precision=3)

### Squeeze-and-excitation is input-dependent channel gating

The gate below exposes its weights for inspection. Changing the input statistics changes which channels are amplified or suppressed; the result is a learned channel interaction, not a semantic explanation.

In [ ]:
class SqueezeExcitation(nn.Module):
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, hidden, 1), nn.ReLU(),
            nn.Conv2d(hidden, channels, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        weights = self.gate(x)
        return x * weights, weights


torch.manual_seed(SEED)
se = SqueezeExcitation(8)
input_a = torch.rand(1, 8, 12, 12)
input_b = input_a.clone(); input_b[:, 2] *= 4; input_b[:, 6] += 2
_, gate_a = se(input_a); _, gate_b = se(input_b)
pd.DataFrame({"channel": range(8), "gate_input_a": gate_a.flatten().detach().numpy(),
              "gate_input_b": gate_b.flatten().detach().numpy()}).style.hide(axis="index").format(precision=3)

## 5. Build five official pretrained encoders

The registry below centralizes construction, weights, head removal, and embedding size. The official weights metadata includes a reference operation estimate and checkpoint file size. Those reference values are useful for orientation, but our timed benchmark uses the same chosen resolution on this machine.

In [ ]:
MODEL_SPECS = OrderedDict({
    "ResNet-18": (resnet18, ResNet18_Weights.DEFAULT),
    "ResNet-50": (resnet50, ResNet50_Weights.DEFAULT),
    "MobileNetV3-Large": (mobilenet_v3_large, MobileNet_V3_Large_Weights.DEFAULT),
    "EfficientNet-B0": (efficientnet_b0, EfficientNet_B0_Weights.DEFAULT),
    "ConvNeXt-Tiny": (convnext_tiny, ConvNeXt_Tiny_Weights.DEFAULT),
})


def remove_head(model: nn.Module, name: str) -> tuple[nn.Module, int]:
    if name.startswith("ResNet"):
        dim = model.fc.in_features
        model.fc = nn.Identity()
    else:
        dim = model.classifier[-1].in_features
        model.classifier[-1] = nn.Identity()
    return model, dim


def build_encoder(name: str) -> tuple[nn.Module, object, int]:
    builder, weights = MODEL_SPECS[name]
    model = builder(weights=weights)
    model, dim = remove_head(model, name)
    return model.to(DEVICE).eval(), weights, dim


model_store, weights_store, embedding_dims = {}, {}, {}
inventory_rows = []
for name in MODEL_SPECS:
    model, weights, dim = build_encoder(name)
    model_store[name], weights_store[name], embedding_dims[name] = model, weights, dim
    inventory_rows.append({
        "model": name,
        "weights": weights.name,
        "embedding_dim": dim,
        "parameters_m": sum(p.numel() for p in model.parameters()) / 1e6,
        "official_ops_g_at_reference_crop": weights.meta.get("_ops", np.nan),
        "official_file_mb": weights.meta.get("_file_size", np.nan),
        "reference_crop": weights.transforms().crop_size[0],
    })
inventory_df = pd.DataFrame(inventory_rows)
assert len(model_store) == 5 and all(not m.training for m in model_store.values())
inventory_df.style.hide(axis="index").format(precision=3)

## 6. Controlled frozen-probe benchmark

Every image is resized to the same benchmark resolution and normalized with the official pretrained-weight statistics. We use each weight package’s interpolation mode. The same balanced logistic-regression configuration is fitted to every frozen embedding set.

This controls the downstream classifier, but not pretraining data/recipe or embedding dimension. Record those remaining differences when interpreting a result.

In [ ]:
SHIFT_FUNCTIONS = {"clean": lambda image: image}


def make_transform(weights, resolution: int):
    official = weights.transforms()
    return transforms.Compose([
        transforms.Resize((resolution, resolution), interpolation=official.interpolation, antialias=True),
        transforms.ToTensor(),
        transforms.Normalize(mean=official.mean, std=official.std),
    ])


class InspectionDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform, shift=lambda image: image):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform
        self.shift = shift

    def __len__(self): return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.path) as image:
            image = self.shift(image.convert("RGB"))
        return self.transform(image), int(row.label_id), index


def extract_features(model, frame, transform, shift=lambda image: image, batch_size=8):
    loader = DataLoader(InspectionDataset(frame, transform, shift), batch_size=batch_size,
                        shuffle=False, num_workers=0)
    features, labels, indices = [], [], []
    model.eval()
    with torch.inference_mode():
        for images, batch_labels, batch_indices in loader:
            output = model(images.to(DEVICE))
            features.append(output.detach().cpu().reshape(len(images), -1).numpy())
            labels.extend(batch_labels.numpy().tolist())
            indices.extend(batch_indices.numpy().tolist())
    order = np.argsort(indices)
    return np.concatenate(features)[order], np.asarray(labels)[order]


def score_predictions(y_true, y_pred, small_mask=None):
    defect_true = (np.asarray(y_true) != 0).astype(int)
    defect_pred = (np.asarray(y_pred) != 0).astype(int)
    result = {
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "defect_recall": recall_score(defect_true, defect_pred, zero_division=0),
    }
    if small_mask is not None and np.asarray(small_mask).any():
        mask = np.asarray(small_mask, dtype=bool)
        result["small_defect_recall"] = recall_score(defect_true[mask], defect_pred[mask], zero_division=0)
    else:
        result["small_defect_recall"] = np.nan
    return result


split_frames = {split: data.query("split == @split").reset_index(drop=True)
                for split in ["train", "val", "test"]}
feature_store, probe_store, prediction_store, benchmark_rows = {}, {}, {}, []

for name, model in model_store.items():
    transform = make_transform(weights_store[name], RUN["benchmark_resolution"])
    feature_store[name] = {}
    for split, frame in split_frames.items():
        feature_store[name][split] = extract_features(model, frame, transform)
    train_x, train_y = feature_store[name]["train"]
    probe = LogisticRegression(max_iter=700, class_weight="balanced", random_state=SEED)
    probe.fit(normalize(train_x), train_y)
    probe_store[name] = probe
    for split in ["val", "test"]:
        features, labels = feature_store[name][split]
        predictions = probe.predict(normalize(features))
        prediction_store[(name, split)] = predictions
        metrics = score_predictions(labels, predictions, split_frames[split].small_defect)
        benchmark_rows.append({"model": name, "split": split, **metrics})

benchmark_df = pd.DataFrame(benchmark_rows)
assert benchmark_df.model.nunique() == 5 and np.isfinite(benchmark_df.macro_f1).all()
display(benchmark_df.pivot(index="model", columns="split", values=["macro_f1", "defect_recall"]).round(3))

best_val_name = benchmark_df.query("split == 'val'").sort_values(
    ["macro_f1", "defect_recall"], ascending=False).iloc[0].model
test_labels = feature_store[best_val_name]["test"][1]
print(f"Validation-selected encoder: {best_val_name}")
print(classification_report(test_labels, prediction_store[(best_val_name, "test")],
                            target_names=CLASS_NAMES, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
test_view = benchmark_df.query("split == 'test'").sort_values("macro_f1")
axes[0].barh(test_view.model, test_view.macro_f1, color="#267D8C")
axes[0].set(xlim=(0, 1), title="Held-out Factory C: macro F1")

cm = confusion_matrix(test_labels, prediction_store[(best_val_name, "test")], labels=range(len(CLASS_NAMES)))
im = axes[1].imshow(cm, cmap="Blues")
axes[1].set(xticks=range(4), yticks=range(4), xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            xlabel="Predicted", ylabel="Actual", title=f"Validation-selected: {best_val_name}")
axes[1].tick_params(axis="x", rotation=35)
for i in range(4):
    for j in range(4): axes[1].text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar(im, ax=axes[1], fraction=.046)
plt.tight_layout(); plt.show()

## 7. Systems profiling: MACs are not latency

The hook-based MAC count is an approximation for `Conv2d` and `Linear` layers only. It excludes normalization, activation, pooling, data movement, and framework overhead. The activation estimate sums produced tensor bytes; it is not allocator peak memory because real runtimes reuse buffers and retain some tensors.

Latency is measured separately for batch 1 and batch 8. We report median, p90, p95, and throughput. CUDA peak allocation is recorded when available. For CPU and MPS, the notebook keeps the transparent hook estimate rather than pretending it measured process peak memory.

In [ ]:
def estimate_macs_and_activations(model: nn.Module, resolution: int):
    totals = {"macs": 0, "activation_bytes": 0, "largest_activation_bytes": 0}
    handles = []

    def hook(module, inputs, output):
        out = output if isinstance(output, torch.Tensor) else output[0]
        batch = out.shape[0]
        elements = out.numel() / batch
        totals["activation_bytes"] += elements * out.element_size()
        totals["largest_activation_bytes"] = max(totals["largest_activation_bytes"], elements * out.element_size())
        if isinstance(module, nn.Conv2d):
            per_output = module.kernel_size[0] * module.kernel_size[1] * module.in_channels / module.groups
            totals["macs"] += elements * per_output
        elif isinstance(module, nn.Linear):
            totals["macs"] += elements * module.in_features

    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            handles.append(module.register_forward_hook(hook))
    with torch.inference_mode():
        model(torch.zeros(1, 3, resolution, resolution, device=DEVICE))
    for handle in handles: handle.remove()
    return totals


def profile_model(model: nn.Module, resolution: int, batch_size: int, repeats: int):
    sample = torch.randn(batch_size, 3, resolution, resolution, device=DEVICE)
    model.eval()
    with torch.inference_mode():
        for _ in range(3): model(sample)
        sync_device()
        if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
        elapsed = []
        for _ in range(repeats):
            sync_device(); start = time.perf_counter()
            model(sample)
            sync_device(); elapsed.append((time.perf_counter() - start) * 1_000)
    return {
        "batch_size": batch_size,
        "median_ms": float(np.median(elapsed)),
        "p90_ms": float(np.percentile(elapsed, 90)),
        "p95_ms": float(np.percentile(elapsed, 95)),
        "throughput_images_s": float(batch_size / (np.median(elapsed) / 1_000)),
        "cuda_peak_mb": (torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == "cuda" else np.nan),
    }


profile_rows = []
for name, model in model_store.items():
    estimates = estimate_macs_and_activations(model, RUN["benchmark_resolution"])
    for batch_size in [1, 8]:
        timed = profile_model(model, RUN["benchmark_resolution"], batch_size, RUN["timing_repeats"])
        profile_rows.append({
            "model": name, "resolution": RUN["benchmark_resolution"],
            "approx_macs_g": estimates["macs"] / 1e9,
            "activation_sum_mb_per_image": estimates["activation_bytes"] / 2**20,
            "largest_activation_mb_per_image": estimates["largest_activation_bytes"] / 2**20,
            **timed,
        })
profile_df = pd.DataFrame(profile_rows)
assert set(profile_df.batch_size) == {1, 8}
profile_df.style.hide(axis="index").format(precision=3)

### Inspect operator-level behavior with `torch.profiler`

Macro timing answers “how long did the encoder take?” The profiler helps answer “where did execution time and memory go?” Profiling itself adds overhead, so its elapsed time is not substituted for the clean latency loop.

In [ ]:
profile_target = model_store["MobileNetV3-Large"]
profiler_input = torch.randn(1, 3, RUN["benchmark_resolution"], RUN["benchmark_resolution"], device=DEVICE)
activities = [ProfilerActivity.CPU]
if DEVICE.type == "cuda": activities.append(ProfilerActivity.CUDA)
try:
    with profile(activities=activities, record_shapes=True, profile_memory=True) as prof:
        with torch.inference_mode(): profile_target(profiler_input)
        sync_device()
    sort_key = "cuda_time_total" if DEVICE.type == "cuda" else "cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=10))
except Exception as exc:
    print(f"Profiler unavailable on this backend: {type(exc).__name__}: {exc}")

## 8. Resolution sweep: does extra spatial evidence pay rent?

We shortlist a small residual baseline, a mobile-oriented model, and a modern high-capacity ConvNet. Each resolution retrains the same probe because the embedding can change with scale. The full run uses the requested 128/224/320 sequence; the default uses 96/128/160 to keep CPU execution practical.

In [ ]:
resolution_models = ["ResNet-18", "MobileNetV3-Large", "ConvNeXt-Tiny"]
resolution_rows = []
for name in resolution_models:
    model, weights = model_store[name], weights_store[name]
    for resolution in RUN["resolution_sweep"]:
        transform = make_transform(weights, resolution)
        train_x, train_y = extract_features(model, split_frames["train"], transform)
        test_x, test_y = extract_features(model, split_frames["test"], transform)
        probe_at_resolution = LogisticRegression(max_iter=700, class_weight="balanced", random_state=SEED)
        probe_at_resolution.fit(normalize(train_x), train_y)
        pred = probe_at_resolution.predict(normalize(test_x))
        scores = score_predictions(test_y, pred, split_frames["test"].small_defect)
        estimates = estimate_macs_and_activations(model, resolution)
        timing = profile_model(model, resolution, batch_size=1, repeats=max(4, RUN["timing_repeats"] // 2))
        resolution_rows.append({
            "model": name, "resolution": resolution, **scores,
            "median_ms_b1": timing["median_ms"],
            "approx_macs_g": estimates["macs"] / 1e9,
            "activation_mb": estimates["activation_bytes"] / 2**20,
        })
resolution_df = pd.DataFrame(resolution_rows)
assert resolution_df.groupby("model").resolution.nunique().eq(3).all()
resolution_df.style.hide(axis="index").format(precision=3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, group in resolution_df.groupby("model"):
    group = group.sort_values("resolution")
    axes[0].plot(group.resolution, group.small_defect_recall, marker="o", label=name)
    axes[1].plot(group.resolution, group.median_ms_b1, marker="o", label=name)
    axes[2].plot(group.resolution, group.activation_mb, marker="o", label=name)
axes[0].set(title="Small-defect recall", xlabel="input side", ylabel="recall", ylim=(0, 1.05))
axes[1].set(title=f"Measured batch-1 latency ({DEVICE.type})", xlabel="input side", ylabel="median ms")
axes[2].set(title="Estimated activation traffic", xlabel="input side", ylabel="summed MiB/image")
for ax in axes: ax.grid(alpha=.25); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 9. Partial fine-tuning: adapt only the last stage

We now change the experiment. ResNet-18 and ConvNeXt-Tiny receive a four-class head, and only their final feature stage plus head is trainable. The assertions make the freeze boundary explicit. One default epoch demonstrates the mechanism; it is not a claim of convergence.

In [ ]:
def build_partial_classifier(name: str):
    builder, weights = MODEL_SPECS[name]
    model = builder(weights=weights)
    for parameter in model.parameters(): parameter.requires_grad = False
    if name == "ResNet-18":
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, len(CLASS_NAMES))
        for parameter in model.layer4.parameters(): parameter.requires_grad = True
        head = model.fc
    elif name == "ConvNeXt-Tiny":
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, len(CLASS_NAMES))
        for parameter in model.features[-1].parameters(): parameter.requires_grad = True
        head = model.classifier[-1]
    else:
        raise ValueError(name)
    return model.to(DEVICE), weights, head


def train_partial(name: str):
    model, weights, head = build_partial_classifier(name)
    transform = make_transform(weights, RUN["benchmark_resolution"])
    train_loader = DataLoader(InspectionDataset(split_frames["train"], transform), batch_size=8,
                              shuffle=True, generator=torch.Generator().manual_seed(SEED), num_workers=0)
    test_loader = DataLoader(InspectionDataset(split_frames["test"], transform), batch_size=8,
                             shuffle=False, num_workers=0)
    trainable = [p for p in model.parameters() if p.requires_grad]
    frozen = [p for p in model.parameters() if not p.requires_grad]
    assert trainable and frozen
    optimizer = torch.optim.AdamW(trainable, lr=2e-4, weight_decay=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    epoch_losses = []
    for _ in range(RUN["fine_tune_epochs"]):
        model.train()
        # Frozen BatchNorm buffers are state too; do not update their running statistics.
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d): module.eval()
        batch_losses = []
        for images, labels, _ in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(images.to(DEVICE)), labels.to(DEVICE))
            loss.backward(); optimizer.step(); batch_losses.append(float(loss.detach().cpu()))
        epoch_losses.append(float(np.mean(batch_losses)))
    model.eval(); predictions, labels_out = [], []
    with torch.inference_mode():
        for images, labels, _ in test_loader:
            predictions.extend(model(images.to(DEVICE)).argmax(1).cpu().numpy())
            labels_out.extend(labels.numpy())
    scores = score_predictions(labels_out, predictions, split_frames["test"].small_defect)
    return model, weights, head, transform, epoch_losses, scores


tuned_store, tuning_rows = {}, []
for name in ["ResNet-18", "ConvNeXt-Tiny"]:
    model, weights, head, transform, losses, scores = train_partial(name)
    tuned_store[name] = {"model": model, "weights": weights, "head": head, "transform": transform}
    frozen_score = benchmark_df.query("model == @name and split == 'test'").iloc[0]
    tuning_rows.append({"model": name, "frozen_probe_f1": frozen_score.macro_f1,
                        "partial_tune_f1": scores["macro_f1"],
                        "partial_tune_defect_recall": scores["defect_recall"],
                        "epoch_losses": [round(x, 4) for x in losses]})
tuning_df = pd.DataFrame(tuning_rows)
tuning_df.style.hide(axis="index").format(precision=3)

## 10. Representation drift after fine-tuning

A forward pre-hook captures the vector entering the new classification head. We compare frozen and tuned embeddings for exactly the same ordered examples. Metrics include per-example cosine drift, nearest-neighbor consistency, and silhouette separation by label and acquisition source.

Source separability is not automatically bad—real factory appearance differs—but unusually strong source clustering can warn that acquisition context dominates the representation.

In [ ]:
def extract_head_inputs(model, head, frame, transform):
    captured = []
    def pre_hook(_module, inputs): captured.append(inputs[0].detach().cpu().reshape(len(inputs[0]), -1))
    handle = head.register_forward_pre_hook(pre_hook)
    loader = DataLoader(InspectionDataset(frame, transform), batch_size=8, shuffle=False, num_workers=0)
    labels, indices = [], []
    model.eval()
    with torch.inference_mode():
        for images, batch_labels, batch_indices in loader:
            model(images.to(DEVICE)); labels.extend(batch_labels.numpy()); indices.extend(batch_indices.numpy())
    handle.remove()
    order = np.argsort(indices)
    return torch.cat(captured).numpy()[order], np.asarray(labels)[order]


drift_rows, embedding_pairs = [], {}
analysis_frame = pd.concat([split_frames["val"], split_frames["test"]], ignore_index=True)
for name, tuned in tuned_store.items():
    frozen_x, frozen_y = extract_features(model_store[name], analysis_frame, tuned["transform"])
    tuned_x, tuned_y = extract_head_inputs(tuned["model"], tuned["head"], analysis_frame, tuned["transform"])
    assert np.array_equal(frozen_y, tuned_y) and frozen_x.shape == tuned_x.shape
    frozen_n, tuned_n = normalize(frozen_x), normalize(tuned_x)
    cosine_drift = 1 - np.sum(frozen_n * tuned_n, axis=1)
    frozen_dist = pairwise_distances(frozen_n, metric="cosine"); np.fill_diagonal(frozen_dist, np.inf)
    tuned_dist = pairwise_distances(tuned_n, metric="cosine"); np.fill_diagonal(tuned_dist, np.inf)
    neighbor_consistency = np.mean(np.argmin(frozen_dist, axis=1) == np.argmin(tuned_dist, axis=1))
    sources = analysis_frame.source.to_numpy()
    drift_rows.append({
        "model": name,
        "mean_cosine_drift": float(cosine_drift.mean()),
        "p90_cosine_drift": float(np.percentile(cosine_drift, 90)),
        "nearest_neighbor_consistency": float(neighbor_consistency),
        "frozen_label_silhouette": silhouette_score(frozen_n, frozen_y, metric="cosine"),
        "tuned_label_silhouette": silhouette_score(tuned_n, tuned_y, metric="cosine"),
        "frozen_source_silhouette": silhouette_score(frozen_n, sources, metric="cosine"),
        "tuned_source_silhouette": silhouette_score(tuned_n, sources, metric="cosine"),
    })
    embedding_pairs[name] = (frozen_n, tuned_n, frozen_y)

drift_df = pd.DataFrame(drift_rows)
drift_df.style.hide(axis="index").format(precision=3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for row, (name, (before, after, labels)) in enumerate(embedding_pairs.items()):
    joined = np.vstack([before, after])
    projected = PCA(n_components=2, random_state=SEED).fit_transform(joined)
    n = len(before)
    for col, (title, points) in enumerate([("frozen", projected[:n]), ("partial tune", projected[n:])]):
        scatter = axes[row, col].scatter(points[:, 0], points[:, 1], c=labels, cmap="tab10", s=18, alpha=.8)
        axes[row, col].set_title(f"{name}: {title}"); axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
plt.suptitle("PCA is a qualitative projection; use the quantitative drift table too")
plt.tight_layout(); plt.show()

## 11. Standardized robustness suite

The frozen probe remains fixed while test images are perturbed. That isolates how much the pretrained representation and fixed decision boundary tolerate each change. Factory C already supplies the held-out-source boundary; the other conditions are applied on top of it.

In [ ]:
def low_light(image): return ImageEnhance.Brightness(image).enhance(0.45)
def blur(image): return image.filter(ImageFilter.GaussianBlur(radius=1.8))
def jpeg(image):
    buffer = io.BytesIO(); image.save(buffer, format="JPEG", quality=28); buffer.seek(0)
    with Image.open(buffer) as loaded: return loaded.convert("RGB")
def sensor_noise(image):
    array = np.asarray(image).astype(np.float32)
    rng = np.random.default_rng(zlib.crc32(image.tobytes()) + SEED)
    return Image.fromarray(np.uint8(np.clip(array + rng.normal(0, 16, array.shape), 0, 255)))
def resolution_loss(image):
    return image.resize((48, 48), Image.Resampling.BILINEAR).resize(image.size, Image.Resampling.BILINEAR)
def combined(image): return jpeg(low_light(blur(image)))


SHIFT_FUNCTIONS = OrderedDict({
    "clean-heldout-source": lambda image: image,
    "blur": blur,
    "low-light": low_light,
    "jpeg-compression": jpeg,
    "sensor-noise": sensor_noise,
    "resolution-loss": resolution_loss,
    "combined-stress": combined,
})

robustness_rows = []
for name, model in model_store.items():
    transform, probe_model = make_transform(weights_store[name], RUN["benchmark_resolution"]), probe_store[name]
    for shift_name, shift_fn in SHIFT_FUNCTIONS.items():
        features, labels = extract_features(model, split_frames["test"], transform, shift=shift_fn)
        predictions = probe_model.predict(normalize(features))
        scores = score_predictions(labels, predictions, split_frames["test"].small_defect)
        robustness_rows.append({"model": name, "condition": shift_name, **scores})
robustness_df = pd.DataFrame(robustness_rows)
clean_lookup = robustness_df.query("condition == 'clean-heldout-source'").set_index("model").macro_f1
robustness_df["f1_drop_from_clean"] = robustness_df.apply(
    lambda row: clean_lookup[row.model] - row.macro_f1, axis=1)
assert robustness_df.groupby("model").condition.nunique().eq(len(SHIFT_FUNCTIONS)).all()
display(robustness_df.pivot(index="model", columns="condition", values="macro_f1").round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
pivot = robustness_df.pivot(index="model", columns="condition", values="macro_f1")
image = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set(xticks=range(len(pivot.columns)), yticks=range(len(pivot.index)),
       xticklabels=pivot.columns, yticklabels=pivot.index, title="Macro F1 by held-out-source stress condition")
ax.tick_params(axis="x", rotation=35)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)): ax.text(j, i, f"{pivot.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(image, ax=ax, label="macro F1"); plt.tight_layout(); plt.show()

## 12. Build Pareto views instead of one leaderboard

We merge frozen-probe quality, averaged shifted quality, file size, and measured execution. A point is marked Pareto-efficient when no other candidate has both higher quality and lower cost. Different plots answer different questions; there is no single architecture ordering independent of objectives.

In [ ]:
def pareto_mask(frame: pd.DataFrame, quality: str, cost: str) -> np.ndarray:
    values = frame[[quality, cost]].to_numpy(float)
    efficient = np.ones(len(frame), dtype=bool)
    for i, (q_i, c_i) in enumerate(values):
        dominated = np.any((values[:, 0] >= q_i) & (values[:, 1] <= c_i) &
                           ((values[:, 0] > q_i) | (values[:, 1] < c_i)))
        efficient[i] = not dominated
    return efficient


test_quality = benchmark_df.query("split == 'test'").set_index("model")
robust_mean = (robustness_df.query("condition != 'clean-heldout-source'")
               .groupby("model").macro_f1.mean().rename("robust_f1"))
b1 = profile_df.query("batch_size == 1").set_index("model")
b8 = profile_df.query("batch_size == 8").set_index("model")
decision_df = inventory_df.set_index("model").join(test_quality[["macro_f1", "defect_recall"]]).join(robust_mean)
decision_df = decision_df.join(b1[["median_ms", "p90_ms", "p95_ms"]].rename(columns=lambda c: f"{c}_b1"))
decision_df = decision_df.join(b8[["throughput_images_s"]].rename(columns={"throughput_images_s": "throughput_b8"}))
decision_df = decision_df.rename(columns={"official_file_mb": "model_mb"}).reset_index()

pareto_specs = [
    ("macro_f1", "median_ms_b1", "Clean F1 vs latency"),
    ("defect_recall", "median_ms_b1", "Defect recall vs latency"),
    ("macro_f1", "model_mb", "Clean F1 vs model size"),
    ("robust_f1", "median_ms_b1", "Robust F1 vs latency"),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (quality, cost, title) in zip(axes.ravel(), pareto_specs):
    mask = pareto_mask(decision_df, quality, cost)
    ax.scatter(decision_df[cost], decision_df[quality], c=np.where(mask, "#E56B3F", "#8A9BA8"), s=75)
    for _, row in decision_df.iterrows(): ax.annotate(row.model, (row[cost], row[quality]), xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.set(xlabel=cost, ylabel=quality, title=title); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()
decision_df.style.hide(axis="index").format(precision=3)

## 13. Evaluate the declared deployment contracts

The selector applies hard constraints first, then the declared lexicographic optimization order. If no model is feasible, it says so instead of quietly relaxing a safety or systems requirement. Because the current notebook may run on a laptop rather than any named target, every recommendation is labeled provisional.

In [ ]:
def feasible_for(frame, hard):
    mask = np.ones(len(frame), dtype=bool)
    for key, value in hard.items():
        if key.endswith("_min"):
            mask &= frame[key[:-4]].to_numpy() >= value
        elif key.endswith("_max"):
            mask &= frame[key[:-4]].to_numpy() <= value
        else:
            raise ValueError(f"Constraint must end in _min or _max: {key}")
    return mask


contract_results = []
for deployment, contract in CONTRACTS.items():
    candidates = decision_df.loc[feasible_for(decision_df, contract["hard"])].copy()
    if candidates.empty:
        contract_results.append({"deployment": deployment, "status": "no feasible measured candidate",
                                 "selected": None, "reason": "At least one hard constraint failed for every model."})
        continue
    sort_columns, ascending = [], []
    for instruction in contract["optimize"]:
        column, direction = instruction.split(":")
        sort_columns.append(column); ascending.append(direction == "min")
    selected = candidates.sort_values(sort_columns, ascending=ascending).iloc[0]
    contract_results.append({"deployment": deployment, "status": "provisional selection",
                             "selected": selected.model,
                             "reason": f"Feasible under {contract['hard']}; ordered by {contract['optimize']}."})

contract_df = pd.DataFrame(contract_results)
contract_df.style.hide(axis="index")

## 14. Save the evidence, not just the conclusion

The decision record includes configuration, environment, contracts, summary measurements, selections, and limitations. This makes a later target-device run comparable and reviewable.

In [ ]:
for name, frame in {
    "model_inventory.csv": inventory_df,
    "frozen_probe_benchmark.csv": benchmark_df,
    "systems_profile.csv": profile_df,
    "resolution_sweep.csv": resolution_df,
    "partial_fine_tuning.csv": tuning_df,
    "representation_drift.csv": drift_df,
    "robustness_suite.csv": robustness_df,
    "deployment_candidates.csv": decision_df,
}.items():
    frame.to_csv(ARTIFACT_DIR / name, index=False)

decision_record = {
    "course": "02-modern-cnn-architectures-efficient-vision",
    "run": RUN,
    "environment": environment,
    "contracts": CONTRACTS,
    "selections": contract_results,
    "candidate_metrics": decision_df.replace({np.nan: None}).to_dict(orient="records"),
    "limitations": [
        "Synthetic images do not estimate production quality.",
        "Latency characterizes only the recorded machine, runtime, dtype, and shapes.",
        "Hook-based MAC and activation figures are transparent approximations, not allocator traces.",
        "The default fine-tuning budget demonstrates mechanics rather than convergence.",
        "A production decision requires target-device, real-camera, safety, and cost validation.",
    ],
}
decision_path = ARTIFACT_DIR / "deployment_decision.json"
decision_path.write_text(json.dumps(decision_record, indent=2), encoding="utf-8")
assert decision_path.exists() and len(contract_results) == 3
print(f"Saved reproducible evidence to {ARTIFACT_DIR}")
print(decision_path.read_text(encoding="utf-8")[:1_500] + "\n...")

## 15. Production tooling review

The eager PyTorch benchmark is the beginning of systems validation, not the end.

| Path | What it can change | Required validation |
| --- | --- | --- |
| `torch.compile` | graph capture, fusion, generated kernels | separate compile/warm-up cost; numerical parity; new latency distribution |
| ONNX Runtime | graph rewrites, provider-specific kernels | export coverage; numerical parity; provider and thread configuration |
| Quantization | weight/activation representation and eligible kernels | per-class quality, calibration data, operator support, real size/latency |
| ExecuTorch or vendor runtime | edge packaging and device kernels | actual device thermals, memory, power, startup, and sustained latency |
| `timm` model registry | architecture and pretrained-recipe breadth | model-card license, transforms, revision pinning, and fair probe setup |

Do not assume a conversion preserves the ranking. Re-run the same deployment contract after each material runtime change.

### Extension exercises

1. Add EfficientNetV2 or ConvNeXt V2 from a maintained registry and document the trust boundary.
2. Export two Pareto candidates to ONNX and test numerical parity before timing.
3. Quantize ResNet-18 and MobileNetV3-Large, then compare file size, defect recall, and p95 latency.
4. Repeat the benchmark with rectangular camera frames and letterboxing rather than square distortion.
5. Replace the synthetic dataset with a licensed industrial dataset and split by acquisition source.
6. Add energy per inference and sustained-temperature behavior to the edge contract.

## 16. What you should now be able to explain without code

Close the notebook and answer these aloud or in writing:

1. How is the degradation problem different from vanishing gradients?
2. Why can an identity shortcut make a deep network easier to optimize?
3. What changes when a residual block downsamples or changes channel count?
4. Why can a model with fewer MACs have higher measured latency?
5. What does a frozen linear probe control, and what differences remain uncontrolled?
6. Why might a higher-resolution model improve small-defect recall but violate an edge contract?
7. How do parameter count, activation memory, batch-1 latency, and throughput differ?
8. Why can partial fine-tuning improve validation F1 while representation reliability gets worse?
9. What does Pareto-efficient mean in this model-selection problem?
10. Why must the winning candidate be retested on the exact target runtime and hardware?

Then write one paragraph for each deployment contract explaining your provisional selection or why no measured candidate was feasible.

## Transition to Course 03

Residual learning, efficient operators, compound scaling, and ConvNeXt make modern CNNs extraordinarily capable. That leaves the next architectural question:

> If modern CNNs are this capable and efficient, why did Vision Transformers become so important?

Course 03 begins with tokens, self-attention, scaling behavior, and the trade-offs that answer that question.